# Final Project for Seminar NN in NLP: How Does Instruction Tuning Affect the Representation Space of Small LLMs?

1. Load the pretrained base model and its tokenizer and test it out a little bit
2. Load the instruction tuning data set and test it out a little bit
3. Instruction tune the full model on the instruction data set and
save frequent checkpoints (mount to google drive for saving)
4. Evaluate instruction following performance for pretrained model and at each checkpoint
5. Evaluate the embedding space by passing a fixed analysis dataset through the checkpoints, extract those embedding and representation layers that I want to investigate and apply several evaluation strategies: geometry, clusters, probing etc
6. Compare instruction following performance and changes in embedding space: do they happen at the same time?

# Prerequisites

In [ ]:
!pip install trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.0/925.0 kB 20.7 MB/s eta 0:00:00


In [ ]:
import trl
import transformers
print(trl.__version__)
print(transformers.__version__)

1.14.0
5.16.1


In [ ]:
import numpy as np
import torch
import pandas as pd

from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer, TrainerCallback

from datasets import load_dataset

from trl import SFTConfig, SFTTrainer

from collections import Counter

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


#Load and check out the pretrained model



In [ ]:
# load the base model, but use the instruct tokenizer,
# both tokenizers have the same vocabulary, but instruct_tokenizer already uses the correct chat template that we need later

# I actually wanted to use setup_chat_format, like this: model, tokenizer = setup_chat_format(model, tokenizer, format="chatml")
# but it does not work anymore with the current trl version, so therefore just use the tokenizer from the instruct model

base_model_name = "HuggingFaceTB/SmolLM2-135M"
model = AutoModelForCausalLM.from_pretrained(base_model_name, dtype=torch.float32).to(device)
base_tokenizer = AutoTokenizer.from_pretrained(base_model_name)

instruct_model_name = "HuggingFaceTB/SmolLM2-135M-Instruct"
instruct_model = AutoModelForCausalLM.from_pretrained(instruct_model_name, dtype=torch.float32).to(device)
instruct_tokenizer = AutoTokenizer.from_pretrained(instruct_model_name)

print("Base tokenizer size:", len(base_tokenizer))
print("Instruct tokenizer size:", len(instruct_tokenizer))
print("Base model embedding size:", model.get_input_embeddings().weight.shape[0])

for token in ["<|im_start|>", "<|im_end|>"]:
    print(token, "base ID:", base_tokenizer.convert_tokens_to_ids(token), "instruct ID:", instruct_tokenizer.convert_tokens_to_ids(token))

print()
print(base_tokenizer.get_vocab() == instruct_tokenizer.get_vocab())

print()
print("Base tokenizer chat template:", base_tokenizer.chat_template)
print("Instruct tokenizer chat template:", instruct_tokenizer.chat_template)

config.json:   0%|          | 0.00/704 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  269MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.66k [00:00<?, ?B/s]

... (11 outputs left out)

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.10M [00:00<?, ?B/s]

Base tokenizer size: 49152
Instruct tokenizer size: 49152
Base model embedding size: 49152
<|im_start|> base ID: 1 instruct ID: 1
<|im_end|> base ID: 2 instruct ID: 2

True

Base tokenizer chat template: None
Instruct tokenizer chat template: {% for message in messages %}{% if loop.first and messages[0]['role'] != 'system' %}{{ '<|im_start|>system
You are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>
' }}{% endif %}{{'<|im_start|>' + message['role'] + '
' + message['content'] + '<|im_end|>' + '
'}}{% endfor %}{% if add_generation_prompt %}{{ '<|im_start|>assistant
' }}{% endif %}


In [ ]:
print(next(model.parameters()).dtype)

torch.float32


In [ ]:
# Check the model architecture
num_params = sum(p.numel() for p in model.parameters())
print(f"Number of parameters: {num_params}")

print(model)

print(model.config)

Number of parameters: 134515008
LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(49152, 576)
    (layers): ModuleList(
      (0-29): 30 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=576, out_features=576, bias=False)
          (k_proj): Linear(in_features=576, out_features=192, bias=False)
          (v_proj): Linear(in_features=576, out_features=192, bias=False)
          (o_proj): Linear(in_features=576, out_features=576, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=576, out_features=1536, bias=False)
          (up_proj): Linear(in_features=576, out_features=1536, bias=False)
          (down_proj): Linear(in_features=1536, out_features=576, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((576,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((576,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((576,), 

In [ ]:
# Check out the tokenizer
#text = "The best musician in the world is"
#text = "Explain why people like to procrastinate"
text = "Summarize what happens in the books about Harry Potter"
inputs = instruct_tokenizer(text, return_tensors="pt").to(device)
print(inputs)
tokens = instruct_tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
print(tokens)

# using the base or instruct tokenizer indeed gives me the exact same results

{'input_ids': tensor([[13764,  3298,   836,   732,  4283,   281,   260,  2905,   563, 15688,
         24595]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], device='cuda:0')}
['Sum', 'mar', 'ize', 'Ġwhat', 'Ġhappens', 'Ġin', 'Ġthe', 'Ġbooks', 'Ġabout', 'ĠHarry', 'ĠPotter']


In [ ]:
# Check out differences between pretrained base model and instruction tuned model
# the instruction tuned model should
# - answer the questions and not just continue the text
# - follow the actual commands
# - produce the answer in a good format, i.e. concise and like an assistant
# - stop at an appropriate point

model.eval()
instruct_model.eval()

def generate_response(model_to_test, prompt, max_new_tokens=100, use_chat_template=False):
    # for many raw instructions the instruct model just directly outputs an end of sequence token, so we use the chat format instead
    if use_chat_template:
        messages = [{"role": "user", "content": prompt}]
        formatted_prompt = instruct_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    else:
        formatted_prompt = prompt

    inputs = instruct_tokenizer(formatted_prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model_to_test.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=instruct_tokenizer.eos_token_id)

    # only decode the generated response and not the input tokens
    input_length = inputs["input_ids"].shape[1]
    generated_text = instruct_tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)

    return generated_text

prompts = [
    "The best musician in the world is",
    "The smartest people always",
    "Explain why people like to procrastinate.",
    "Summarize what happens in the books about Harry Potter.",
    "What is the most used word in the english language?",
    "Are you a language model?",
    "List three examples of renewable energy.",
    "Write a poem with eleven words."
]

for prompt in prompts:
    base_response = generate_response(model, prompt)
    instruct_response = generate_response(instruct_model, prompt, use_chat_template=True)
    print("Prompt:", prompt)
    print()
    print("Base model:", base_response)
    print()
    print("Instruct model:", instruct_response)
    print()
    print("-----")
    print()

Prompt: The best musician in the world is

Base model:  a musician who can play a song in a way that is not only beautiful, but also effective.

The best musician in the world is a musician who can play a song in a way that is not only beautiful, but also effective.

The best musician in the world is a musician who can play a song in a way that is not only beautiful, but also effective.

The best musician in the world is a musician who can play a song in a way that is not

Instruct model: The best musician in the world is a topic of much debate among musicians and critics. While it's true that many great musicians have achieved greatness, there are those who have struggled to find their place in the world of music.

In the realm of music, there are many talented musicians who have achieved success, but there are also many who have struggled to find their place in the world of music. Some musicians have achieved great success, while others have struggled to find their place in the world

In [ ]:
# do random sampling
prompt = "As my father always used to say"
inputs = instruct_tokenizer(prompt, return_tensors="pt").to(device)
with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=100, do_sample=True, temperature=0.8, top_p=0.95)
generated_text = instruct_tokenizer.decode(outputs[0], skip_special_tokens=True)
print(generated_text)

As my father always used to say, the more you look after your children, the better they will grow up, and the more they will enjoy the good life.

The old proverb

And this is the truth; the children of the gods have been blessed with

the power to love the people, the children of the gods have been blessed with

the power to do the menial tasks.

But the great evil is that the gods have not the power to do the menial tasks.




In [ ]:
# make sure I can access the hidden states (important for representation space analysis later)
text = "Explain why people like to procrastinate"
inputs = instruct_tokenizer(text, return_tensors="pt").to(device)
model.eval()
with torch.no_grad():
    outputs = model(**inputs, output_hidden_states=True)

hidden_states = outputs.hidden_states
print("Number of hidden states:", len(hidden_states))
for i, hidden_state in enumerate(hidden_states):
    print(f"Layer {i}: {hidden_state.shape}")

# layer 0 is the initial embedding and the rest are the transformer layers that follow

Number of hidden states: 31
Layer 0: torch.Size([1, 7, 576])
Layer 1: torch.Size([1, 7, 576])
Layer 2: torch.Size([1, 7, 576])
Layer 3: torch.Size([1, 7, 576])
Layer 4: torch.Size([1, 7, 576])
Layer 5: torch.Size([1, 7, 576])
Layer 6: torch.Size([1, 7, 576])
Layer 7: torch.Size([1, 7, 576])
Layer 8: torch.Size([1, 7, 576])
Layer 9: torch.Size([1, 7, 576])
Layer 10: torch.Size([1, 7, 576])
Layer 11: torch.Size([1, 7, 576])
Layer 12: torch.Size([1, 7, 576])
Layer 13: torch.Size([1, 7, 576])
Layer 14: torch.Size([1, 7, 576])
Layer 15: torch.Size([1, 7, 576])
Layer 16: torch.Size([1, 7, 576])
Layer 17: torch.Size([1, 7, 576])
Layer 18: torch.Size([1, 7, 576])
Layer 19: torch.Size([1, 7, 576])
Layer 20: torch.Size([1, 7, 576])
Layer 21: torch.Size([1, 7, 576])
Layer 22: torch.Size([1, 7, 576])
Layer 23: torch.Size([1, 7, 576])
Layer 24: torch.Size([1, 7, 576])
Layer 25: torch.Size([1, 7, 576])
Layer 26: torch.Size([1, 7, 576])
Layer 27: torch.Size([1, 7, 576])
Layer 28: torch.Size([1, 7, 57

In [ ]:
# check for one particular token that can then later be compared between pre-trained version and instruction tuned versions
token_index = 2
representation = hidden_states[-1][0, token_index]
print(representation.shape)

torch.Size([576])


In [ ]:
# memory
print(f"Allocated memory: {torch.cuda.memory_allocated(device)} bytes")
print(f"Reserved memory: {torch.cuda.memory_reserved(device)} bytes")

Allocated memory: 558895616 bytes
Reserved memory: 1107296256 bytes


#Load and Check out the Instruction Tuning Dataset

In [ ]:
dataset = load_dataset("databricks/databricks-dolly-15k", split="train")
# smollm corpus: https://huggingface.co/collections/HuggingFaceTB/smollm
print(dataset)

README.md:   0%|          | 0.00/8.20k [00:00<?, ?B/s]

databricks-dolly-15k.jsonl: reconstructing file:   0%|          |  0.00B / 13.1MB            

databricks-dolly-15k.jsonl: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/15011 [00:00<?, ? examples/s]

Dataset({
    features: ['instruction', 'context', 'response', 'category'],
    num_rows: 15011
})


In [ ]:
print("Columns:", dataset.column_names)
print("Features", dataset.features)
print("Categories:", dataset.unique("category"))

category_amounts = Counter(dataset["category"])
for category, amount in category_amounts.items():
    print(f"{category}: {amount}")

Columns: ['instruction', 'context', 'response', 'category']
Features {'instruction': Value('string'), 'context': Value('string'), 'response': Value('string'), 'category': Value('string')}
Categories: ['closed_qa', 'classification', 'open_qa', 'information_extraction', 'brainstorming', 'general_qa', 'summarization', 'creative_writing']
closed_qa: 1773
classification: 2136
open_qa: 3742
information_extraction: 1506
brainstorming: 1766
general_qa: 2191
summarization: 1188
creative_writing: 709


In [ ]:
example = dataset[57]
for key, value in example.items():
    print(f"{key}: {value}")

# not all examples have a context

instruction: What is your favorite strip from the comic Calvin and Hobbes?
context: 
response: Calvin and Hobbes is an iconic strip that is regarded as one of the best comic strips in history. Individual preferences may vary and it's difficult to find the publishing rate of individual strips throughout its ten year syndication.
category: general_qa


# Prepocess Dataset for Training

Huggingface SFT Trainer Explanations here: https://huggingface.co/docs/trl/sft_trainer

Alignment Handbook where the chat template is specified: https://github.com/huggingface/alignment-handbook/blob/main/recipes/smollm2/sft/config_smol.yaml

Expected data set type is either language modeling or prompt-completion and then either standard or conversational => I use conversational prompt-completion format, so that we match their instruction tuning setup the SmolLM2 authors used in their experiments and we can make the loss only apply on the response

In [ ]:
# formatting function for the examples, this is not a chat template!!
# use the conversational prompt-completion format

def format_example(example):
    instruction = example["instruction"].strip()
    context = example["context"].strip()
    response = example["response"].strip()

    if context:
        user_message = (f"{instruction}\n\nContext:\n{context}")
    else:
        user_message = instruction

    return {"prompt": [{"role": "user", "content": user_message}],
            "completion": [{"role": "assistant", "content": response}]}

In [ ]:
split_dataset = dataset.train_test_split(test_size=0.1, seed=90)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]
print(len(train_dataset))
print(len(eval_dataset))

13509
1502


In [ ]:
train_dataset = train_dataset.map(format_example, remove_columns=train_dataset.column_names)
eval_dataset = eval_dataset.map(format_example, remove_columns=eval_dataset.column_names)
print(train_dataset[10])

Map:   0%|          | 0/13509 [00:00<?, ? examples/s]

Map:   0%|          | 0/1502 [00:00<?, ? examples/s]

{'prompt': [{'content': 'What is the mission of the Federal Bureau of Investigation?', 'role': 'user'}], 'completion': [{'content': 'The Federal Bureau of Investigation (FBI) is the domestic intelligence and security service of the United States and its principal federal law enforcement agency. Operating under the jurisdiction of the United States Department of Justice, the FBI is also a member of the U.S. Intelligence Community and reports to both the Attorney General and the Director of National Intelligence.  A leading U.S. counterterrorism, counterintelligence, and criminal investigative organization, the FBI has jurisdiction over violations of more than 200 categories of federal crimes.', 'role': 'assistant'}]}


In [ ]:
example = train_dataset[0]
messages = example["prompt"] + example["completion"]
formatted = instruct_tokenizer.apply_chat_template(messages, tokenize=False)
print(repr(formatted[:512]))

'<|im_start|>system\nYou are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>\n<|im_start|>user\nExtract the list of European Union member countries\n\nContext:\nThe European Union (EU) is a political and economic union of 27 member states that are signatories to the founding treaties of the union and thereby share in the privileges and obligations of membership. They have agreed by the treaties to share their own sovereignty through the institutions of the European Union in some, but not all'


In [ ]:
tokens = instruct_tokenizer.apply_chat_template(messages, tokenize=True, return_dict=True)
print(tokens.keys())
print("Sequence length:", len(tokens["input_ids"]))

KeysView({'input_ids': [1, 9690, 198, 2683, 359, 253, 5356, 5646, 11173, 3365, 3511, 308, 34519, 28, 7018, 411, 407, 19712, 8182, 2, 198, 1, 4093, 198, 34683, 260, 1398, 282, 3284, 4353, 4028, 2075, 198, 198, 17548, 42, 198, 504, 3284, 4353, 365, 25953, 25, 314, 253, 2356, 284, 2536, 8964, 282, 216, 34, 39, 4028, 2496, 338, 359, 914, 12616, 288, 260, 14465, 19785, 282, 260, 8964, 284, 6914, 2419, 281, 260, 19195, 284, 13803, 282, 13405, 30, 1069, 457, 6454, 411, 260, 19785, 288, 2419, 480, 1038, 14532, 738, 260, 4679, 282, 260, 3284, 4353, 281, 634, 28, 564, 441, 511, 28, 3260, 282, 1693, 30, 2888, 5536, 1251, 3387, 39921, 281, 260, 5299, 327, 260, 8964, 288, 6074, 634, 4057, 43, 327, 1449, 28, 6675, 3884, 359, 1135, 411, 11645, 4559, 12448, 30, 1216, 13803, 284, 4347, 282, 14532, 1127, 260, 7905, 365, 23895, 5124, 288, 347, 393, 1180, 276, 780, 25, 919, 357, 2116, 1486, 3113, 13540, 28, 347, 357, 553, 3251, 624, 1038, 3425, 1686, 527, 411, 260, 11920, 282, 260, 14465, 19785, 314, 1062

In [ ]:
print("Pad token:", instruct_tokenizer.pad_token)
print("EOS token:", instruct_tokenizer.eos_token)

Pad token: <|im_end|>
EOS token: <|im_end|>


In [ ]:
# check how long the instructions and responses are (amount of characters) to determine a good sequence length
def get_chat_token_length(example):
    messages = example["prompt"] + example["completion"]
    tokenized = instruct_tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=False, return_dict=True)
    return len(tokenized["input_ids"])

train_lengths = [get_chat_token_length(example) for example in train_dataset]

print("Minimum:", np.min(train_lengths))
print("Maximum:", np.max(train_lengths))
print("Mean:", np.mean(train_lengths))
print("Median:", np.median(train_lengths))
print("90th percentile:", np.percentile(train_lengths, 90))
print("95th percentile:", np.percentile(train_lengths, 95))

for max_length in [128,256,512,1024]:
    fraction_truncated = np.mean(np.array(train_lengths) > max_length)
    print(f"Max length = {max_length}, then {fraction_truncated * 100}% examples would be truncated")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (8789 > 8192). Running this sequence through the model will result in indexing errors


Minimum: 35
Maximum: 8789
Mean: 215.0780960840921
Median: 137.0
90th percentile: 429.2000000000007
95th percentile: 604.5999999999985
Max length = 128, then 53.282996520837955% examples would be truncated
Max length = 256, then 24.206084832334% examples would be truncated
Max length = 512, then 7.217410615145459% examples would be truncated
Max length = 1024, then 1.6137389888222669% examples would be truncated


In [ ]:
def get_prompt_token_length(example):
    tokenized = instruct_tokenizer.apply_chat_template(example["prompt"], tokenize=True, add_generation_prompt=True, return_dict=True,)
    return len(tokenized["input_ids"])


prompt_lengths = [get_prompt_token_length(example) for example in train_dataset]

print("Mean:", np.mean(prompt_lengths))
print("Median:", np.median(prompt_lengths))
print("90th percentile:", np.percentile(prompt_lengths, 90))
print("95th percentile:", np.percentile(prompt_lengths, 95))
print("Maximum:", np.max(prompt_lengths))

for max_length in [256, 512, 768, 1024]:
    fraction = np.mean(np.array(prompt_lengths) >= max_length)
    print(f"{fraction * 100:.2f}% of prompts alone are already >= {max_length} tokens")

Mean: 130.99511436819898
Median: 46.0
90th percentile: 311.2000000000007
95th percentile: 454.0
Maximum: 8499
13.72% of prompts alone are already >= 256 tokens
4.02% of prompts alone are already >= 512 tokens
1.61% of prompts alone are already >= 768 tokens
0.93% of prompts alone are already >= 1024 tokens


# Instruction Tuning of the Model

with an effective batch size of 16 and about 13.500 training examples this gives us 13.500/16 = 844 optimizer steps per epoch, i.e. about 2531 optimizer steps for three epochs

saving one checkpoint of tuned SmolLM2-135M needs roughly 520MB of storage, i.e. I can even save 90 checkpoints (but probably way less for the bigger model)

save 10 -> 25 -> 50 -> 100 -> 200 -> 400 -> 600 -> 844 (Epoch 1) -> 1.5 -> Epoch 2 -> 2.5 -> Epoch 3

In [ ]:
import inspect
print(inspect.signature(SFTConfig))

(output_dir: str | None = None, per_device_train_batch_size: int = 8, num_train_epochs: float = 3.0, max_steps: int = -1, learning_rate: float = 2e-05, lr_scheduler_type: transformers.trainer_utils.SchedulerType | str = 'linear', lr_scheduler_kwargs: dict | str | None = None, warmup_steps: float = 0, optim: transformers.training_args.OptimizerNames | str = 'adamw_torch_fused', optim_args: str | None = None, weight_decay: float = 0.0, adam_beta1: float = 0.9, adam_beta2: float = 0.999, adam_epsilon: float = 1e-08, optim_target_modules: None | str | list[str] = None, gradient_accumulation_steps: int = 1, average_tokens_across_devices: bool = True, max_grad_norm: float = 1.0, label_smoothing_factor: float = 0.0, bf16: bool | None = None, fp16: bool = False, bf16_full_eval: bool = False, fp16_full_eval: bool = False, tf32: bool | None = None, gradient_checkpointing: bool = True, gradient_checkpointing_kwargs: dict[str, typing.Any] | str | None = None, torch_compile: bool = False, torch_com

In [ ]:
training_args = SFTConfig(
    output_dir="/content/drive/MyDrive/NLP_Final_Project_Instruction_Tuning/smollm2_135M_dolly",

    max_length=512, # chosen due to full token length: not truncate too many examples, but also not make the max length so huge
    truncation_mode="keep_start",
    packing=False, # so that we specifically have: one example = one training sequence
    completion_only_loss=True, # so that the loss is only calculated on the actual response (the completion)

    num_train_epochs=3,
    per_device_train_batch_size=4, # effective batch size: 4x4=16 examples per optimizer update
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,

    learning_rate=2e-5,
    lr_scheduler_type="cosine", # lr gradually declines over training
    warmup_steps=0.05,
    weight_decay=0.01,

    fp16=True,
    bf16=False, # handle mixed precision, otherwise I got an error

    eval_strategy="no", # previously used "steps", but now I am using the callback
    #eval_steps=100, # compute validation loss every 100 optimizer steps, i.e. a total of 25 times during 3 epochs
    logging_strategy="steps",
    logging_steps=25,
    save_strategy="no", # previously used "steps", but now I am using the callback to save exactly the checkpoints that I want to have
    #save_steps=100,
    #save_total_limit=None,

    seed=90, # for reproduceability

    #report_to=None,
)

In [ ]:
# save more checkpoints in the beginning to see fast changes and less checkpoints later to save storage space
# see callbacks here: https://huggingface.co/docs/transformers/en/main_classes/callback?utm_source=chatgpt.com
save_steps = [10,25,50,100,200,400,600,800,1200,1600,2000,2400]
eval_steps = [10,25,50,100,200,300,400,500,600,700,800,900,1000,1100,1200,1300,1400,1500,1600,1700,1800,1900,2000,2100,2200,2300,2400,2500]

class SaveAndEvalCheckpointCallback(TrainerCallback):
    def __init__(self, save_steps, eval_steps):
        self.save_steps = save_steps
        self.eval_steps = eval_steps

    def on_step_end(self, args, state, control, **kwargs):
        # save a checkpoint at the specified steps
        if state.global_step in self.save_steps:
            control.should_save = True
        # compute validation loss at the specified steps
        if state.global_step in self.eval_steps:
            control.should_evaluate = True
        return control

save_and_eval_checkpoint_callback = SaveAndEvalCheckpointCallback(save_steps, eval_steps)

In [ ]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=instruct_tokenizer,
    callbacks=[save_and_eval_checkpoint_callback]
)

In [ ]:
# get the exact number of optimizer steps per epoch, so that I can put a checkpoint directly at the end of each epoch
train_dataloader = trainer.get_train_dataloader()
batches_per_epoch = len(train_dataloader)
optimizer_steps_per_epoch = (batches_per_epoch + training_args.gradient_accumulation_steps - 1) // training_args.gradient_accumulation_steps
print(f"Batches per epoch: {batches_per_epoch}")
print(f"Optimizer steps per epoch: {optimizer_steps_per_epoch}")

Batches per epoch: 3242
Optimizer steps per epoch: 811


In [ ]:
print("Model dtype", next(model.parameters()).dtype)
print("Trainer device", trainer.args.device)
print("fp16", trainer.args.fp16)
print("bf16", trainer.args.bf16)
print("GPU:", torch.cuda.get_device_name(0))

Model dtype torch.float32
Trainer device cuda:0
fp16 True
bf16 False
GPU: Tesla T4


In [ ]:
initial_eval = trainer.evaluate()
print(initial_eval)

Training Loss,Validation Loss,Step,Entropy,Num Tokens,Mean Token Accuracy
No log,2.607635,0,2.560749,0.000000,0.531716


{'eval_loss': 2.607635498046875, 'eval_entropy': 2.560748628301964, 'eval_num_tokens': 0.0, 'eval_mean_token_accuracy': 0.5317164706556421}


In [ ]:
# TRAINING IS HERE!!!
train_result = trainer.train()

# TODO: what to do in case of colab disconnection

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1, 'pad_token_id': 2}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
10,No log,2.583800,2.542778,28715.000000,0.532993
25,2.571588,2.450612,2.451590,73419.000000,0.540005
50,2.432061,2.280473,2.312875,144035.000000,0.549665
75,2.347919,No Log,No Log,No Log,No Log
100,2.279676,2.156883,2.147587,281098.000000,0.557218
125,2.188605,No Log,No Log,No Log,No Log
150,2.161831,No Log,No Log,No Log,No Log
175,2.166158,No Log,No Log,No Log,No Log
200,2.153658,2.064516,2.082949,561959.000000,0.569132
225,2.207951,No Log,No Log,No Log,No Log


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (4 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
# save final model
final_model_dir = "/content/drive/MyDrive/NLP_Final_Project_Instruction_Tuning/smollm2_135M_dolly/final_model"
trainer.save_model(final_model_dir)
instruct_tokenizer.save_pretrained(final_model_dir)
trainer.save_state()

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
# get validation losses after training
log_df = pd.DataFrame(trainer.state.log_history)
log_df.to_csv("/content/drive/MyDrive/NLP_Final_Project_Instruction_Tuning/smollm2_135M_dolly/training_log.csv", index=False)

eval_df = log_df[log_df["eval_loss"].notna()][["step", "epoch", "eval_loss"]]
train_loss_df = log_df[log_df["loss"].notna()][["step", "epoch", "loss"]]

In [ ]:
train_loss_df

,step,epoch,loss
1,25,0.030845,2.571588
3,50,0.061690,2.432061
5,75,0.092535,2.347919
6,100,0.123381,2.279676
8,125,0.154226,2.188605
...,...,...,...
118,2325,2.867366,2.014670
119,2350,2.898211,1.984936
120,2375,2.929056,1.963797
121,2400,2.959901,1.862486


In [ ]:
eval_df

,step,epoch,eval_loss
0,10,0.012338,2.583800
2,25,0.030845,2.450612
4,50,0.061690,2.280473
7,100,0.123381,2.156883
12,200,0.246761,2.064516
17,300,0.370142,2.041012
22,400,0.493523,2.030874
27,500,0.616903,2.022218
32,600,0.740284,2.015889
37,700,0.863664,2.010259


In [ ]:
# evaluate dolly validation loss of the published instruct model
published_instruct_trainer = SFTTrainer(
    model=instruct_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=instruct_tokenizer,
    callbacks=[save_and_eval_checkpoint_callback]
)

published_instruct_eval = published_instruct_trainer.evaluate()
print(published_instruct_eval)

Tokenizing train dataset:   0%|          | 0/13509 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/13509 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/13509 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/13509 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/1502 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/1502 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/1502 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/1502 [00:00<?, ? examples/s]

Training Loss,Validation Loss,Step,Entropy,Num Tokens,Mean Token Accuracy
No log,2.291578,0,1.804114,0.000000,0.543218


{'eval_loss': 2.2915782928466797, 'eval_entropy': 1.8041139628913594, 'eval_num_tokens': 0.0, 'eval_mean_token_accuracy': 0.5432178074303096}


# Performance Evaluation by checking the generated outputs

In [ ]:
prompts = [
    # simple text continuation
    "The best musician in the world is",
    "Someday the sun will explode because",
    # facutal questions
    "What is the capital of France?",
    "What is a Computer?",
    # open QA
    "Explain why the sky is blue.",
    "Explain why humans need sleep.",
    # listing
    "List three examples of renewable energy.",
    "Give two reasons why exercise is beneficial",
    # summarization
    "Summarize what happens in the books about Harry Potter.",
    "Summarize this text in one sentence: Alice and Bob both enjoy math and encriptions. They are even mentioned by name in many text books and lectures. However, they value their privacy and would rather not be mentioned by name. Therefore, they plan on initializing a lawsuit.",
    # classification
    "Classify the Harry Potter in one genre: Romance, Fiction, History, Non-fiction",
    # yes / no
    "Is the sky blue?",
    "Is the pope a catholic?",
    # simple reasoning
    "Alice has 12 apples. She eats one and gives one to Bob. How many apples has she left?",
    # one precise constraint
    "Answer with exactly three words: who are you?",
    # creative instruction
    "Write a poem with eleven words.",
    # assistant style
    "Give me three tips for doing a computer science project.",
    "Tell me the one most important thing for a successful workout session."
    # something the model cannot know
    "What did I eat this morning for breakfast?"
]

In [ ]:
checkpoint10_model = AutoModelForCausalLM.from_pretrained("/content/drive/MyDrive/NLP_Final_Project_Instruction_Tuning/smollm2_135M_dolly/checkpoint-10", dtype=torch.float32).to(device)
checkpoint100_model = AutoModelForCausalLM.from_pretrained("/content/drive/MyDrive/NLP_Final_Project_Instruction_Tuning/smollm2_135M_dolly/checkpoint-100", dtype=torch.float32).to(device)
checkpointfinal_model = AutoModelForCausalLM.from_pretrained("/content/drive/MyDrive/NLP_Final_Project_Instruction_Tuning/smollm2_135M_dolly/final_model", dtype=torch.float32).to(device)

for prompt in prompts:
    base_response = generate_response(model, prompt)
    checkpoint10_response = generate_response(checkpoint10_model, prompt, use_chat_template=True)
    checkpoint100_response = generate_response(checkpoint100_model, prompt, use_chat_template=True)
    checkpointfinal_response = generate_response(checkpointfinal_model, prompt, use_chat_template=True)
    instruct_response = generate_response(instruct_model, prompt, use_chat_template=True)
    print("Prompt:", prompt)
    print()
    print("Base model:", base_response)
    print()
    print("Checkpoint 10:", checkpoint10_response)
    print()
    print("Checkpoint 100:", checkpoint100_response)
    print()
    print("Checkpoint final:", checkpointfinal_response)
    print()
    print("Instruct model:", instruct_response)
    print()
    print("-----")
    print()

Loading weights:   0%|          | 0/272 [00:03<?, ?it/s]

Loading weights:   0%|          | 0/272 [00:03<?, ?it/s]

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Prompt: The best musician in the world is

Base model:  a musician who can play a song in a way that is not only beautiful, but also effective.

The best musician in the world is a musician who can play a song in a way that is not only beautiful, but also effective.


Checkpoint 10: The best singer in the world ispite
piteassistant
The best singer in the world ispite
piteassistant
The best singer in the world ispite
piteassistant
The best singer in the world ispite
piteass

Checkpoint 100: The best musician in the world is a musician.

The best musician in the world is a musician.

The best musician in the world is a musician.

The best musician in the world is a musician.

The best

Checkpoint final: The best musician in the world is a question that has been asked by many people.  The answer is that it depends on the person.  Some people say that the best musician is Mozart, while others say that the best musician is Beethoven. 

Instruct model: The best musician in the world is a top

# Performance Evaluation with IFEval

use lm-evaluation-harness from https://github.com/eleutherai/lm-evaluation-harness

it implements many evaluation tasks for generative language models and is commonly used for Huggingface models

In [ ]:
!pip install -q "lm_eval[hf,ifeval]"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.1/64.1 kB 4.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 32.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.1/91.1 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 7.9 MB/s eta 0:00:00


In [ ]:
import lm_eval
import importlib.metadata

import subprocess
import os
import shutil
import re
import json
from pathlib import Path

print(importlib.metadata.version("lm_eval"))

0.4.13


In [ ]:
# !lm-eval -h
!lm-eval ls tasks # prints a huge list of available tasks

Die letzten 5000 Zeilen der Streamingausgabe wurden abgeschnitten.
|masakhanews_yor_prompt_4                                                              |lm_eval/tasks/afrobench/masakhanews/prompt_4/masakhanews_yor.yaml                                                                                               |multiple_choice      |
|masakhanews_yor_prompt_5                                                              |lm_eval/tasks/afrobench/masakhanews/prompt_5/masakhanews_yor.yaml                                                                                               |multiple_choice      |
|masakhapos_bam_prompt_1                                                               |lm_eval/tasks/afrobench/masakhapos/prompt_1/masakhapos_bam.yaml                                                                                                 |generate_until       |
|masakhapos_bam_prompt_2                                                               |lm_eval/tasks/afrobench/masak

In [ ]:
def run_ifeval(model_path, output_path, tokenizer_path="HuggingFaceTB/SmolLM2-135M-Instruct", batch_size=8, limit=None, apply_chat_template=False):
    os.makedirs(output_path, exist_ok=True)

    model_args = (f"pretrained={model_path},tokenizer={tokenizer_path}")

    # command taken from the github repo of lm-evaluation-harness
    command = ["lm_eval", "--model", "hf", "--model_args", model_args, "--tasks", "ifeval", "--device", f"{device}:0", "--batch_size", str(batch_size), "--log_samples", "--output_path", output_path]
    # log_samples saves the generated responses

    if apply_chat_template:
        command += ["--apply_chat_template"]

    if limit is not None:
        # "limit" determines the amount of prompts that are used for evaluation, put None for the final experiments (IFEval contains 541 prompts)
        command += ["--limit", str(limit)]

    result = subprocess.run(command, text=True, capture_output=True)
    print(result.stdout)

    if result.returncode != 0:
        print("Error:", result.stderr)
        return result

    # lm-eval-harness creates an additional subdirectory, therefore we move the files up one level
    subdirectories = [os.path.join(output_path, d) for d in os.listdir(output_path) if os.path.isdir(os.path.join(output_path, d))]
    print(subdirectories)
    for subdirectory in subdirectories:
        for file in os.listdir(subdirectory):
            source = os.path.join(subdirectory, file)
            destination = os.path.join(output_path, file)
            shutil.move(source, destination)
        os.rmdir(subdirectory)

    return result

In [ ]:
# run ifeval on different checkpoints
# I could run the ifeval evaluation in a loop, but it takes about 1h or 1h15min, so I will very likely loose GPU connection at some point
base_directory = "/content/drive/MyDrive/NLP_Final_Project_Instruction_Tuning/smollm2_135M_dolly"
checkpoint_to_evaluate = "checkpoint-1600"
result = run_ifeval(model_path=f"{base_directory}/{checkpoint_to_evaluate}",
                    output_path=f"{base_directory}/ifeval_results/{checkpoint_to_evaluate}",
                    tokenizer_path="HuggingFaceTB/SmolLM2-135M-Instruct",
                    batch_size=8, limit=None, apply_chat_template=True)

hf ({'pretrained': '/content/drive/MyDrive/NLP_Final_Project_Instruction_Tuning/smollm2_135M_dolly/checkpoint-1600', 'tokenizer': 'HuggingFaceTB/SmolLM2-135M-Instruct'}), gen_kwargs: ({}), limit: None, num_fewshot: None, batch_size: 8
|Tasks |Version|Filter|n-shot|        Metric         |   |Value |   |Stderr|
|------|------:|------|-----:|-----------------------|---|-----:|---|------|
|ifeval|      4|none  |     0|inst_level_loose_acc   |↑  |0.2338|±  |   N/A|
|      |       |none  |     0|inst_level_strict_acc  |↑  |0.2182|±  |   N/A|
|      |       |none  |     0|prompt_level_loose_acc |↑  |0.1257|±  |0.0143|
|      |       |none  |     0|prompt_level_strict_acc|↑  |0.1165|±  |0.0138|


['/content/drive/MyDrive/NLP_Final_Project_Instruction_Tuning/smollm2_135M_dolly/ifeval_results/checkpoint-1600/__content__drive__MyDrive__NLP_Final_Project_Instruction_Tuning__smollm2_135M_dolly__checkpoint-1600']


In [ ]:
# run ifeval on the base model and on the instruction tuned model that was published by SmolLM
base_directory = "/content/drive/MyDrive/NLP_Final_Project_Instruction_Tuning/smollm2_135M_dolly"
smolLM2_135_instruct = "HuggingFaceTB/SmolLM2-135M-Instruct"
smolLM2_135_base = "HuggingFaceTB/SmolLM2-135M"
result = run_ifeval(model_path=smolLM2_135_base,
                    output_path=f"{base_directory}/ifeval_results/base",
                    tokenizer_path="HuggingFaceTB/SmolLM2-135M-Instruct",
                    batch_size=8, apply_chat_template=True)

hf ({'pretrained': 'HuggingFaceTB/SmolLM2-135M', 'tokenizer': 'HuggingFaceTB/SmolLM2-135M-Instruct'}), gen_kwargs: ({}), limit: None, num_fewshot: None, batch_size: 8
|Tasks |Version|Filter|n-shot|        Metric         |   |Value |   |Stderr|
|------|------:|------|-----:|-----------------------|---|-----:|---|------|
|ifeval|      4|none  |     0|inst_level_loose_acc   |↑  |0.3981|±  |   N/A|
|      |       |none  |     0|inst_level_strict_acc  |↑  |0.3549|±  |   N/A|
|      |       |none  |     0|prompt_level_loose_acc |↑  |0.2717|±  |0.0191|
|      |       |none  |     0|prompt_level_strict_acc|↑  |0.2237|±  |0.0179|


['/content/drive/MyDrive/NLP_Final_Project_Instruction_Tuning/smollm2_135M_dolly/ifeval_results/base/HuggingFaceTB__SmolLM2-135M']


In [ ]:
# inspect the logged samples to make sure everything works properly
import json
from pprint import pprint

samples = []

base_directory = "/content/drive/MyDrive/NLP_Final_Project_Instruction_Tuning/smollm2_135M_dolly"
with open(f"{base_directory}/ifeval_results/published-instruct_apply-chat-template/samples_ifeval_2026-08-22T09-55-14.706629.jsonl", "r") as f:
    for line in f:
        samples.append(json.loads(line))

print(f"Number of samples: {len(samples)}")
print(samples[0].keys())
pprint(samples[0])

Number of samples: 541
dict_keys(['doc_id', 'doc', 'target', 'arguments', 'resps', 'filtered_resps', 'filter', 'metrics', 'doc_hash', 'prompt_hash', 'target_hash', 'prompt_level_strict_acc', 'inst_level_strict_acc', 'prompt_level_loose_acc', 'inst_level_loose_acc'])
{'arguments': {'gen_args_0': {'arg_0': '<|im_start|>system\n'
                                       'You are a helpful AI assistant named '
                                       'SmolLM, trained by Hugging '
                                       'Face<|im_end|>\n'
                                       '<|im_start|>user\n'
                                       'Write a 300+ word summary of the '
                                       'wikipedia page '
                                       '"https://en.wikipedia.org/wiki/Raymond_III,_Count_of_Tripoli". '
                                       'Do not use any commas and highlight at '
                                       'least 3 sections that has titles in '
         

In [ ]:
# make a sorted list of all checkpoints that are saved in the Google Drive folder
all_checkpoints = [checkpoint for checkpoint in os.listdir(base_directory) if re.fullmatch(r"checkpoint-\d+", checkpoint)]
all_checkpoints = sorted(all_checkpoints, key=lambda x: int(x.split("-")[1]))
# I could also add the final model and the base model and the SmolLM2 Instruct, but I can also just run these separately
print(all_checkpoints)

ifeval_directory = os.path.join(base_directory, "ifeval_results")
os.makedirs(ifeval_directory, exist_ok=True)

['checkpoint-10', 'checkpoint-25', 'checkpoint-50', 'checkpoint-100', 'checkpoint-200', 'checkpoint-400', 'checkpoint-600', 'checkpoint-800', 'checkpoint-1200', 'checkpoint-1600', 'checkpoint-2000', 'checkpoint-2400']


In [ ]:
# aggregate all ifeval results into one dataframe
all_checkpoints = ["base", "checkpoint-10", "checkpoint-50", "checkpoint-100", "checkpoint-400", "checkpoint-800", "checkpoint-1600", "final_model", "published-instruct_apply-chat-template"]
rows = []
for checkpoint in all_checkpoints:
    ifeval_directory_checkpoint = Path(ifeval_directory) / checkpoint
    result_file = list(ifeval_directory_checkpoint.glob("results*.json"))[0]
    with open(result_file, "r") as f:
        result_dict = json.load(f)
    scores = result_dict["results"]["ifeval"]

    rows.append({"model": checkpoint,
                 "instuction_strict": scores["inst_level_strict_acc,none"],
                 "prompt_strict": scores["prompt_level_strict_acc,none"],
                 "instuction_loose": scores["inst_level_loose_acc,none"],
                 "prompt_loose": scores["prompt_level_loose_acc,none"],
    })

ifeval_df = pd.DataFrame(rows)
ifeval_df

# instruction-level accuracy is higher because a response can satisfy some constraints while failing others
# prompt-level accuracy requires satisfying all constraints associated with the prompt


,model,instuction_strict,prompt_strict,instuction_loose,prompt_loose
0,base,0.354916,0.223660,0.398082,0.271719
1,checkpoint-10,0.357314,0.223660,0.396882,0.266174
2,checkpoint-50,0.287770,0.182994,0.316547,0.208872
3,checkpoint-100,0.270983,0.157116,0.286571,0.168207
4,checkpoint-400,0.243405,0.109057,0.254197,0.116451
5,checkpoint-800,0.225420,0.114603,0.237410,0.120148
6,checkpoint-1600,0.218225,0.116451,0.233813,0.125693
7,final_model,0.219424,0.114603,0.235012,0.123845
8,published-instruct_apply-chat-template,0.357314,0.227357,0.380096,0.242144
